# Nädal 7: RFM kliendisegmenteerimine

## Roll B: Data Cleaning

Minu alaülesanne on puhastada Roll A poolt ette valmistatud liidetud andmestik: eemaldada duplikaadid, käsitleda kriitilised NULL väärtused, parsida kuupäevad, eemaldada vigased hinnaread ja koostada puhastusraport. Lõpptulemus on `df`, mis on valmis Roll C RFM analüüsiks.

## Sisend: Roll A väljund

Grupitöös tuleb see lahter asendada Roll A liidetud DataFrame'iga. Portfoolios on notebook tehtud iseseisvalt käivitatavaks: kui `df` ei ole juba olemas, loetakse müügi- ja kliendiandmed CSV-failidest ning tehakse sama merge `customer_id` põhjal.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


def find_file(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    raise FileNotFoundError(f'Faili ei leitud. Kontrollitud asukohad: {candidates}')


if 'df' not in globals():
    sales_path = find_file([
        '../../datasets/clean/sales.csv',
        '../datasets/clean/sales.csv',
        '../../SQL/sales_rows.csv',
        'SQL/sales_rows.csv',
    ])
    customers_path = find_file([
        '../../datasets/clean/customers.csv',
        '../datasets/clean/customers.csv',
        '../../SQL/customers_rows.csv',
        'SQL/customers_rows.csv',
    ])

    df_sales = pd.read_csv(sales_path)
    df_customers = pd.read_csv(customers_path)
    df = pd.merge(df_sales, df_customers, on='customer_id', how='left')

print('Roll A liidetud DataFrame shape:', df.shape)
print('\nVeergude tüübid:')
print(df.dtypes)
display(df.head())

required_columns = ['customer_id', 'sale_date', 'total_price', 'email']
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Puuduvad kohustuslikud veerud: {missing_columns}')

## Roll B puhastus

Puhastan andmed juhendi järgi: duplikaadid, kriitilised NULL-id, kuupäevad, positiivne `total_price` ning raport lõpliku kuju, klientide arvu ja kuupäevavahemikuga.

In [ ]:
print('Esialgne shape:', df.shape)

initial_rows = len(df)
duplicates_before = df.duplicated().sum()
print('Duplikaadid enne eemaldamist:', duplicates_before)

df_clean = df.drop_duplicates().copy()
duplicates_after = df_clean.duplicated().sum()
print('Duplikaadid pärast eemaldamist:', duplicates_after)

print('\nNULL-id enne puhastamist:')
nulls_before = df_clean.isnull().sum()
print(nulls_before[nulls_before > 0].sort_values(ascending=False))

critical_columns = ['customer_id', 'sale_date', 'total_price']
critical_nulls_before = df_clean[critical_columns].isnull().sum()
df_clean = df_clean.dropna(subset=critical_columns).copy()

df_clean['sale_date'] = pd.to_datetime(df_clean['sale_date'], errors='coerce')
invalid_dates = df_clean['sale_date'].isna().sum()
df_clean = df_clean.dropna(subset=['sale_date']).copy()

df_clean['total_price'] = pd.to_numeric(df_clean['total_price'], errors='coerce')
invalid_prices = df_clean['total_price'].isna().sum()
df_clean = df_clean.dropna(subset=['total_price']).copy()

non_positive_prices = (df_clean['total_price'] <= 0).sum()
df_clean = df_clean[df_clean['total_price'] > 0].copy()

analysis_end_date = pd.to_datetime('2025-02-28')
rows_after_price_cleaning = len(df_clean)
df_clean = df_clean[df_clean['sale_date'] <= analysis_end_date].copy()
rows_after_date_filter = len(df_clean)

final_rows = len(df_clean)
removed_rows = initial_rows - final_rows

print('\nPuhastusraport')
print('-' * 40)
print(f'Algne ridade arv: {initial_rows}')
print(f'Eemaldatud duplikaate: {duplicates_before}')
print('Kriitilised NULL-id enne eemaldamist:')
print(critical_nulls_before)
print(f'Vigaseid kuupäevi pärast parsimist: {invalid_dates}')
print(f'Mittearvulisi total_price väärtusi: {invalid_prices}')
print(f'Mittepositiivseid total_price väärtusi: {non_positive_prices}')
print(f'Parast 2025-02-28 kuupaevafiltrit eemaldatud ridu: {rows_after_price_cleaning - rows_after_date_filter}')
print(f'Eemaldatud ridu kokku: {removed_rows}')
print(f'Lõplik shape: {df_clean.shape}')
print(f'Unikaalseid kliente: {df_clean["customer_id"].nunique()}')
print(f'Kuupäevavahemik: {df_clean["sale_date"].min().date()} kuni {df_clean["sale_date"].max().date()}')

print('\nKontroll: kriitiliste veergude NULL-id pärast puhastamist')
print(df_clean[critical_columns].isnull().sum())
print('\nsale_date tüüp:', df_clean['sale_date'].dtype)
print('Min total_price:', df_clean['total_price'].min())

# Roll C sisend: puhastatud DataFrame kannab edasi sama nime df.
df = df_clean.reset_index(drop=True)
display(df.head())

## Roll C jaoks üleantav väljund

Puhastatud DataFrame on muutujas `df`. See sisaldab ainult ridu, kus `customer_id`, `sale_date` ja `total_price` on olemas, `sale_date` on `datetime` tüüpi ning `total_price` on positiivne arv.